# params-iterable-vs-groups — worked example 3: Group dicts get default lr without mutating caller input

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `params-iterable-vs-groups`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When the user passes a list of group dicts, some groups may be missing an `'lr'` key — they rely on the optimizer to fill in a default. The tricky correctness requirement is that the optimizer must not mutate the caller's original dicts, because the caller might reuse them. The solution is to make a shallow copy of each group dict with `dict(group)` and then add the `'lr'` key only to the copy.

## Worked solution

**Step 1 — Detect dict-list mode.** `isinstance(materialized[0], dict)` is True when the caller passed a list of group dicts (multi-group API).

**Step 2 — Shallow-copy each dict.** `g = dict(group)` creates a new dict with the same key-value pairs. We work with `g`, not `group`, so modifications don't reach the caller's data structure.

**Step 3 — Fill missing lr.** `if 'lr' not in g: g['lr'] = default_lr`. This adds the key only on the copy, not the original.

**Step 4 — Collect and return.** Append each `g` to `out` and return. The caller's list is untouched, which we verify by checking the original group dict still lacks an `'lr'` key.

In [ ]:
import torch as t

def normalize_group_dicts(params, default_lr):
    """Normalize a list of group dicts; fill missing lr without mutating caller's dicts."""
    materialized = list(params)
    if not materialized:
        raise ValueError('optimizer got an empty parameter list')
    first = materialized[0]
    if isinstance(first, t.Tensor):
        return [{'params': materialized, 'lr': default_lr}]
    if isinstance(first, dict):
        out = []
        for group in materialized:
            g = dict(group)          # shallow copy — do NOT modify the original
            if 'lr' not in g:
                g['lr'] = default_lr
            out.append(g)
        return out
    raise TypeError(f'Unexpected type: {type(first).__name__}')

t.manual_seed(42)
p_fast = [t.randn(4, 4), t.randn(4)]
p_slow = [t.randn(8, 4)]

# Caller builds their own group dicts — intentionally NO 'lr' in second group
caller_groups = [
    {'params': p_fast, 'lr': 1e-3},   # has lr
    {'params': p_slow},               # no lr — needs default
]

result = normalize_group_dicts(caller_groups, default_lr=1e-4)

print(f'Returned groups: {len(result)}')                          # 2
print(f'Group 0 lr: {result[0]["lr"]}')                           # 0.001
print(f'Group 1 lr: {result[1]["lr"]}')                           # 0.0001 (filled in)
print(f'Caller group 1 unchanged: {"lr" not in caller_groups[1]}') # True — no mutation